
# 06\_algoritme\_renditjeje

## Modelim në Fizikë  
### Renditja e faqeve (*page ranking*) në sisteme informacioni

Ky notebook trajton **algoritmet e renditjes së faqeve**, pra problemin e vendosjes në rend prioritar të dokumenteve ose faqeve në ueb në përgjigje të një kërkimi ose brenda një rrjeti lidhjesh.

Në versionin fillestar të këtij kapitulli mund të krijohej përshtypja se “renditja” i referohej renditjes klasike të listave numerike.  
Në këtë version të korrigjuar, fokusi është te **renditja e faqeve**, siç përdoret në:

- motorë kërkimi,
- biblioteka digjitale,
- rrjete citimesh shkencore,
- sisteme rekomandimi,
- analiza e rrjeteve komplekse.

Ky problem është thelbësor edhe në shkencë, sepse shumë struktura informative mund të modelohen si **grafe**:
- artikuj shkencorë dhe citimet ndërmjet tyre;
- faqe interneti dhe hiper-lidhjet;
- dokumente teknike dhe referencat e brendshme;
- nyje informacioni në rrjete semantike.

Në këtë notebook do të ndërtojmë:
1. një model bazë të renditjes me **PageRank**;
2. një version më realist me:
   - relevancë ndaj pyetjes,
   - peshim të lidhjeve,
   - freski të dokumentit,
   - cilësi burimore,
   - sinjale përdoruesi,
   - penalizim të spam-it;
3. eksperimente numerike dhe krahasime.



## Objektivat mësimore

Në fund të këtij materiali, studenti duhet të jetë në gjendje të:

1. kuptojë problemin e renditjes së faqeve si problem i modelimit mbi grafe;
2. shpjegojë idenë probabilistike të algoritmit **PageRank**;
3. implementojë PageRank në mënyrë iterative;
4. kuptojë kufizimet e një algoritmi vetëm me lidhje;
5. ndërtojë një model më realist të renditjes duke kombinuar disa sinjale;
6. analizojë numerikisht ndikimin e parametrave në renditje.


In [ ]:

import math
import re
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt



---

# 1. Hyrje në algoritmet e renditjes së faqeve

## 1.1 Pse renditja e faqeve është e rëndësishme?

Në një koleksion të madh dokumentesh, përdoruesi zakonisht nuk kërkon thjesht **të gjejë** dokumente, por të marrë në krye të listës dokumentet **më të rëndësishme**, **më relevante** dhe **më të besueshme**.

Në terma formalë, na duhet një funksion vlerësimi
\[
\mathrm{Score}(q, d),
\]
ku:
- \(q\) është pyetja (*query*),
- \(d\) është dokumenti ose faqja.

Pastaj dokumentet renditen sipas kësaj score-je.

## 1.2 Dy ide themelore

Ekzistojnë dy familje të mëdha sinjalesh:

### (a) Sinjale strukturore
Këto burojnë nga vetë rrjeti i lidhjeve:
- sa faqe referojnë një faqe të caktuar;
- sa autoritative janë ato faqe;
- si shpërndahet rëndësia në graf.

Kjo është fusha klasike e **PageRank**.

### (b) Sinjale përmbajtësore dhe operative
Këto burojnë nga vetë dokumenti dhe sjellja e përdoruesve:
- përputhja semantike me pyetjen;
- freskia kohore;
- cilësia editoriale;
- norma e klikimit;
- koha e qëndrimit;
- sinjale anti-spam.

Në sistemet reale, renditja moderne është zakonisht një **kombinim shumë-sinjalësh**.

## 1.3 Kompleksiteti dhe kostoja llogaritëse

Në renditjen e faqeve, pyetja nuk është vetëm “sa operacione duhen për një listë me gjatësi \(n\)”, por:
- sa i madh është grafi \(G=(V,E)\);
- sa iterime duhen për konvergjencë;
- sa kushton llogaritja e relevancës ndaj pyetjes;
- a mund të parapërllogariten disa score offline?

Për PageRank klasik, një iterim ka kosto afërsisht proporcionale me numrin e lidhjeve:
\[
O(|E|).
\]
Nëse duhen \(k\) iterime, kostoja totale është:
\[
O(k|E|).
\]

Ky është një shembull tipik ku analiza algoritmike ndërthuret me modelimin matematik të proceseve stokastike.



---

# 2. Algoritmi bazë: PageRank

## 2.1 Ideja themelore

PageRank mund të interpretohet si modeli i një **surfer-i të rastësishëm** që:
- me probabilitet \(d\) ndjek një lidhje nga faqja aktuale;
- me probabilitet \(1-d\) “teleportohet” në një faqe tjetër.

Kështu, rëndësia e një faqeje nuk varet vetëm nga numri i lidhjeve hyrëse, por nga **rëndësia e faqeve që e referojnë**.

Formula iterative standarde është:
\[
PR_{t+1}(i) = \frac{1-d}{N} + d \sum_{j \to i} \frac{PR_t(j)}{L(j)},
\]
ku:
- \(N\) është numri i faqeve,
- \(d\) është faktori i zbehjes (*damping factor*),
- \(L(j)\) është numri i lidhjeve dalëse nga faqja \(j\).

## 2.2 Interpretim fizik dhe probabilistik

Ky model lidhet me:
- zinxhirët e Markovit,
- shpërndarjen stacionare,
- procese të difuzionit në rrjet.

Prandaj është veçanërisht i përshtatshëm në një kurs të modelimit shkencor: problemi i renditjes shndërrohet në problem të një dinamike probabilistike mbi graf.


In [ ]:

# Koleksion i vogël dokumentesh sintetikë
pages = {
    "P1": {
        "title": "Modelimi numerik në fizikë",
        "text": "modelim fizik simulim numerik metoda diferenciale stabilitet gabim",
        "links": ["P2", "P3", "P4"],
        "age_days": 50,
        "quality": 0.92,
        "ctr": 0.18,
        "dwell_time": 210,
        "spam_score": 0.02,
    },
    "P2": {
        "title": "Metoda Monte Carlo",
        "text": "monte carlo fizik probabilitet integrim rastesor simulim",
        "links": ["P1", "P3", "P5"],
        "age_days": 120,
        "quality": 0.88,
        "ctr": 0.16,
        "dwell_time": 190,
        "spam_score": 0.03,
    },
    "P3": {
        "title": "Analiza e stabilitetit numerik",
        "text": "stabilitet numerik gabim skema iterime konvergence fizik",
        "links": ["P1", "P4"],
        "age_days": 20,
        "quality": 0.94,
        "ctr": 0.22,
        "dwell_time": 260,
        "spam_score": 0.01,
    },
    "P4": {
        "title": "Përpunimi i sinjaleve",
        "text": "sinjal transformata fourier filtrim spekter frekuence zhurme",
        "links": ["P1", "P3", "P6"],
        "age_days": 300,
        "quality": 0.83,
        "ctr": 0.12,
        "dwell_time": 160,
        "spam_score": 0.05,
    },
    "P5": {
        "title": "Rrjete komplekse dhe grafe",
        "text": "graf nyje lidhje centralitet pagerank rrjet kompleks",
        "links": ["P1", "P3", "P6", "P7"],
        "age_days": 35,
        "quality": 0.90,
        "ctr": 0.20,
        "dwell_time": 240,
        "spam_score": 0.02,
    },
    "P6": {
        "title": "Bazat e motorëve të kërkimit",
        "text": "kerkimi informacionit relevanca indeksim ranking query dokument",
        "links": ["P1", "P3", "P5"],
        "age_days": 10,
        "quality": 0.95,
        "ctr": 0.25,
        "dwell_time": 280,
        "spam_score": 0.01,
    },
    "P7": {
        "title": "Faqe komerciale me densitet të lartë fjalësh kyçe",
        "text": "ranking ranking ranking klikoni tani ofertë klikim ranking query query",
        "links": ["P6"],
        "age_days": 5,
        "quality": 0.35,
        "ctr": 0.08,
        "dwell_time": 35,
        "spam_score": 0.70,
    },
}
page_ids = list(pages.keys())
page_ids


In [ ]:

def build_transition_matrix(pages):
    ids = list(pages.keys())
    index = {pid: i for i, pid in enumerate(ids)}
    n = len(ids)
    M = np.zeros((n, n))  # M[i,j] = probabiliteti që masa nga j shkon te i

    for source in ids:
        j = index[source]
        links = [target for target in pages[source]["links"] if target in index]

        if len(links) == 0:
            # nyje pa dalje: shpërndarje uniforme
            M[:, j] = 1 / n
        else:
            prob = 1 / len(links)
            for target in links:
                i = index[target]
                M[i, j] = prob
    return ids, index, M

ids, index, M = build_transition_matrix(pages)
M


In [ ]:

def pagerank_basic(pages, damping=0.85, max_iter=100, tol=1e-10):
    ids, index, M = build_transition_matrix(pages)
    n = len(ids)

    pr = np.ones(n) / n
    history = [pr.copy()]

    for _ in range(max_iter):
        new_pr = (1 - damping) / n + damping * (M @ pr)
        history.append(new_pr.copy())

        if np.linalg.norm(new_pr - pr, 1) < tol:
            pr = new_pr
            break
        pr = new_pr

    return ids, pr, np.array(history)

ids, pr_basic, history_basic = pagerank_basic(pages)
for pid, score in sorted(zip(ids, pr_basic), key=lambda x: x[1], reverse=True):
    print(f"{pid}: {score:.4f} | {pages[pid]['title']}")


In [ ]:

plt.figure(figsize=(9, 5))
for i, pid in enumerate(ids):
    plt.plot(history_basic[:, i], marker='o', label=pid)
plt.xlabel("Iterimi")
plt.ylabel("Vlera e PageRank")
plt.title("Konvergjenca e PageRank bazik")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



## 2.3 Kufizimet e PageRank bazik

PageRank klasik është elegant dhe shumë i rëndësishëm historikisht, por ka kufizime serioze në sisteme reale:

1. **nuk përdor pyetjen e përdoruesit**;  
2. **nuk lexon përmbajtjen e dokumentit**;  
3. **nuk merr parasysh freskinë**;  
4. **nuk dallon drejtpërdrejt cilësinë editoriale**;  
5. **mund të ndikohet nga struktura artificiale e lidhjeve**.

Prandaj, në sisteme moderne, PageRank zakonisht përdoret si **një komponent** dhe jo si kriteri i vetëm.



---

# 3. Algoritmi i përmirësuar: një model më realist i renditjes

Do të ndërtojmë tani një model më realist, i cili kombinon disa burime informacioni.

## 3.1 Komponentët e modelit

Për një pyetje \(q\) dhe një dokument \(d\), do të përdorim:

### (a) Autoriteti global
\[
PR(d)
\]
i marrë nga PageRank ose një variant i tij.

### (b) Relevanca tekstuale
\[
Rel(q,d)
\]
që mat sa mirë përputhet dokumenti me pyetjen.

### (c) Freskia
\[
Fresh(d)
\]
që u jep përparësi dokumenteve më të reja kur ky faktor është i rëndësishëm.

### (d) Cilësia e burimit
\[
Qual(d)
\]
që mund të përfaqësojë reputacionin, korrektësinë ose autoritetin editorial.

### (e) Sinjalet e përdoruesit
\[
Beh(d)
\]
p.sh. norma e klikimit dhe koha e qëndrimit.

### (f) Penalizimi i spam-it
\[
Spam(d)
\]
për të zbritur score-n e dokumenteve të dyshimta.

## 3.2 Formula e kombinuar

Një model i thjeshtuar, por realist, mund të jetë:
\[
Score(q,d)=
\alpha\,PR(d)+
\beta\,Rel(q,d)+
\gamma\,Fresh(d)+
\delta\,Qual(d)+
\varepsilon\,Beh(d)-
\zeta\,Spam(d).
\]

Në praktikë, koeficientët \(\alpha,\beta,\gamma,\delta,\varepsilon,\zeta\) mund të mësohen nga të dhënat.  
Në këtë notebook do t’i zgjedhim në mënyrë të arsyetuar për të ndërtuar një model demonstrues.


In [ ]:

def tokenize(text):
    return re.findall(r"[a-zA-ZçëÇË]+", text.lower())

def build_vocabulary(pages):
    vocab = set()
    for page in pages.values():
        vocab.update(tokenize(page["title"]))
        vocab.update(tokenize(page["text"]))
    return sorted(vocab)

vocab = build_vocabulary(pages)
len(vocab), vocab[:20]


In [ ]:

def tf_vector(tokens, vocab_index):
    counts = Counter(tokens)
    vec = np.zeros(len(vocab_index))
    total = sum(counts.values()) or 1
    for term, count in counts.items():
        if term in vocab_index:
            vec[vocab_index[term]] = count / total
    return vec

def idf_from_pages(pages, vocab):
    vocab_index = {term: i for i, term in enumerate(vocab)}
    n_docs = len(pages)
    df = np.zeros(len(vocab))

    for page in pages.values():
        terms = set(tokenize(page["title"] + " " + page["text"]))
        for t in terms:
            if t in vocab_index:
                df[vocab_index[t]] += 1

    idf = np.log((1 + n_docs) / (1 + df)) + 1
    return vocab_index, idf

vocab_index, idf = idf_from_pages(pages, vocab)

def tfidf_vector(text, vocab_index, idf):
    tf = tf_vector(tokenize(text), vocab_index)
    return tf * idf

def cosine_similarity(a, b):
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

doc_vectors = {}
for pid, page in pages.items():
    full_text = page["title"] + " " + page["text"]
    doc_vectors[pid] = tfidf_vector(full_text, vocab_index, idf)


In [ ]:

def normalize_dict(score_dict):
    values = np.array(list(score_dict.values()), dtype=float)
    vmin = values.min()
    vmax = values.max()
    if np.isclose(vmax, vmin):
        return {k: 0.0 for k in score_dict}
    return {k: (v - vmin) / (vmax - vmin) for k, v in score_dict.items()}

def freshness_score(age_days, tau=120):
    # Sa më i vogël age_days, aq më e madhe freskia
    return math.exp(-age_days / tau)

def behavior_score(ctr, dwell_time, dwell_ref=300):
    dwell_norm = min(dwell_time / dwell_ref, 1.0)
    return 0.5 * ctr / 0.25 + 0.5 * dwell_norm

def relevance_scores(query, pages, doc_vectors, vocab_index, idf):
    q_vec = tfidf_vector(query, vocab_index, idf)
    scores = {}
    for pid in pages:
        scores[pid] = cosine_similarity(q_vec, doc_vectors[pid])
    return scores

def page_feature_scores(pages):
    fresh = {pid: freshness_score(page["age_days"]) for pid, page in pages.items()}
    qual = {pid: page["quality"] for pid, page in pages.items()}
    beh  = {pid: behavior_score(page["ctr"], page["dwell_time"]) for pid, page in pages.items()}
    spam = {pid: page["spam_score"] for pid, page in pages.items()}
    return normalize_dict(fresh), normalize_dict(qual), normalize_dict(beh), normalize_dict(spam)



## 3.3 PageRank i personalizuar sipas pyetjes

Në vend që teleportimi të jetë uniform, mund të orientojmë surfer-in e rastësishëm drejt dokumenteve që janë më relevante për pyetjen.  
Kjo jep një variant të quajtur shpesh **personalized PageRank**.

Formula bëhet:
\[
PR_{t+1} = (1-d)\,v_q + d\,M\,PR_t,
\]
ku \(v_q\) është vektori i personalizimit të ndërtuar nga pyetja.

Kjo ide e bën modelin më realist: autoriteti nuk është më krejtësisht global, por pjesërisht i orientuar nga konteksti i kërkimit.


In [ ]:

def personalized_pagerank(pages, personalization, damping=0.85, max_iter=100, tol=1e-10):
    ids, index, M = build_transition_matrix(pages)
    n = len(ids)

    v = np.array([personalization[pid] for pid in ids], dtype=float)
    if np.allclose(v.sum(), 0):
        v = np.ones(n) / n
    else:
        v = v / v.sum()

    pr = np.ones(n) / n

    for _ in range(max_iter):
        new_pr = (1 - damping) * v + damping * (M @ pr)
        if np.linalg.norm(new_pr - pr, 1) < tol:
            pr = new_pr
            break
        pr = new_pr

    return ids, pr

query = "modelim numerik fizik ranking"
rel = relevance_scores(query, pages, doc_vectors, vocab_index, idf)
rel_norm = normalize_dict(rel)

ids, pr_personalized = personalized_pagerank(pages, rel_norm, damping=0.85)

print("Relevanca ndaj pyetjes:")
for pid, score in sorted(rel_norm.items(), key=lambda x: x[1], reverse=True):
    print(f"{pid}: {score:.4f} | {pages[pid]['title']}")


In [ ]:

def weighted_link_transition_matrix(pages, authority_hint=None):
    ids = list(pages.keys())
    index = {pid: i for i, pid in enumerate(ids)}
    n = len(ids)
    M = np.zeros((n, n))

    if authority_hint is None:
        authority_hint = {pid: 1.0 for pid in ids}

    for source in ids:
        j = index[source]
        links = [target for target in pages[source]["links"] if target in pages]

        if len(links) == 0:
            M[:, j] = 1 / n
            continue

        weights = []
        for target in links:
            w = 0.7 * authority_hint.get(target, 1.0) + 0.3 * pages[target]["quality"]
            weights.append(max(w, 1e-12))

        weights = np.array(weights, dtype=float)
        weights = weights / weights.sum()

        for target, w in zip(links, weights):
            i = index[target]
            M[i, j] = w
    return ids, index, M

def pagerank_weighted(pages, personalization=None, damping=0.85, max_iter=100, tol=1e-10):
    # Fillimisht marrim një PageRank bazik si authority hint
    ids0, pr0, _ = pagerank_basic(pages, damping=damping, max_iter=max_iter, tol=tol)
    authority_hint = {pid: score for pid, score in zip(ids0, pr0)}

    ids, index, M = weighted_link_transition_matrix(pages, authority_hint=authority_hint)
    n = len(ids)

    if personalization is None:
        v = np.ones(n) / n
    else:
        v = np.array([personalization.get(pid, 0.0) for pid in ids], dtype=float)
        if np.allclose(v.sum(), 0):
            v = np.ones(n) / n
        else:
            v = v / v.sum()

    pr = np.ones(n) / n
    for _ in range(max_iter):
        new_pr = (1 - damping) * v + damping * (M @ pr)
        if np.linalg.norm(new_pr - pr, 1) < tol:
            pr = new_pr
            break
        pr = new_pr
    return ids, pr

ids, pr_weighted = pagerank_weighted(pages, personalization=rel_norm)
{pid: float(score) for pid, score in zip(ids, pr_weighted)}



## 3.4 Score finale e renditjes

Tani do të kombinojmë:
- PageRank bazik,
- Personalized / Weighted PageRank,
- relevancën tekstuale,
- freskinë,
- cilësinë,
- sinjalet e përdoruesit,
- penalizimin e spam-it.

Kjo nuk pretendon të jetë një motor kërkimi industrial, por është një model **dukshëm më realist** sesa PageRank klasik i izoluar.


In [ ]:

def final_ranking(query, pages, alpha=0.15, beta=0.35, gamma=0.10, delta=0.15, epsilon=0.15, zeta=0.10):
    rel = relevance_scores(query, pages, doc_vectors, vocab_index, idf)
    rel_norm = normalize_dict(rel)

    ids_basic, pr_basic, _ = pagerank_basic(pages)
    pr_basic_dict = normalize_dict({pid: score for pid, score in zip(ids_basic, pr_basic)})

    ids_w, pr_weighted = pagerank_weighted(pages, personalization=rel_norm)
    pr_weighted_dict = normalize_dict({pid: score for pid, score in zip(ids_w, pr_weighted)})

    fresh, qual, beh, spam = page_feature_scores(pages)

    scores = {}
    components = {}
    for pid in pages:
        authority = 0.4 * pr_basic_dict[pid] + 0.6 * pr_weighted_dict[pid]
        score = (
            alpha * authority
            + beta * rel_norm[pid]
            + gamma * fresh[pid]
            + delta * qual[pid]
            + epsilon * beh[pid]
            - zeta * spam[pid]
        )
        scores[pid] = score
        components[pid] = {
            "authority": authority,
            "relevance": rel_norm[pid],
            "freshness": fresh[pid],
            "quality": qual[pid],
            "behavior": beh[pid],
            "spam_penalty": spam[pid],
            "final_score": score,
        }
    return scores, components

query = "modelim numerik fizik ranking"
scores, components = final_ranking(query, pages)

for pid, score in sorted(scores.items(), key=lambda x: x[1], reverse=True):
    print(f"{pid}: {score:.4f} | {pages[pid]['title']}")



---

# 4. Krahasimi i algoritmeve dhe modeleve

Në këtë seksion do të krahasojmë:

1. **PageRank bazik**  
2. **PageRank i personalizuar / i peshëzuar**  
3. **Score finale shumë-sinjalëshe**

Qëllimi është të kuptohet se si ndryshon renditja kur kalojmë nga një model strukturor në një model më realist.


In [ ]:

def ranking_table(query):
    rel = normalize_dict(relevance_scores(query, pages, doc_vectors, vocab_index, idf))
    ids_basic, pr_basic, _ = pagerank_basic(pages)
    basic = normalize_dict({pid: score for pid, score in zip(ids_basic, pr_basic)})

    ids_p, pr_p = personalized_pagerank(pages, rel)
    personal = normalize_dict({pid: score for pid, score in zip(ids_p, pr_p)})

    scores, components = final_ranking(query, pages)
    final_norm = normalize_dict(scores)

    rows = []
    for pid in pages:
        rows.append({
            "pid": pid,
            "title": pages[pid]["title"],
            "PageRank_basic": basic[pid],
            "Personalized_PR": personal[pid],
            "Relevance": rel[pid],
            "Final_score": final_norm[pid],
        })
    return rows

rows = ranking_table(query)
for row in sorted(rows, key=lambda r: r["Final_score"], reverse=True):
    print(row)


In [ ]:

# Grafik krahasues i tri score-ve kryesore
rows_sorted = sorted(rows, key=lambda r: r["Final_score"], reverse=True)
labels = [r["pid"] for r in rows_sorted]
basic_vals = [r["PageRank_basic"] for r in rows_sorted]
pers_vals = [r["Personalized_PR"] for r in rows_sorted]
final_vals = [r["Final_score"] for r in rows_sorted]

x = np.arange(len(labels))
width = 0.25

plt.figure(figsize=(11, 6))
plt.bar(x - width, basic_vals, width=width, label="PageRank bazik")
plt.bar(x, pers_vals, width=width, label="Personalized PR")
plt.bar(x + width, final_vals, width=width, label="Score finale")

plt.xticks(x, labels)
plt.ylabel("Score e normalizuar")
plt.title("Krahasimi i modeleve të renditjes së faqeve")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.show()


In [ ]:

# Analizë e kontributeve për faqet kryesore
top_pages = [pid for pid, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:5]]

for pid in top_pages:
    print(f"\n{pid} | {pages[pid]['title']}")
    for key, value in components[pid].items():
        print(f"  {key:12s}: {value:.4f}")



## 4.1 Interpretim

Një dokument mund të mos jetë:
- më i lidhuri në rrjet,
- por të jetë më relevant për pyetjen;
- më i ri;
- me cilësi më të lartë;
- me sjellje më të mirë nga përdoruesit.

Po kështu, një dokument me shumë përsëritje fjalësh kyçe mund të marrë një relevancë sipërfaqësore, por të penalizohet nga:
- cilësia e ulët,
- koha e shkurtër e qëndrimit,
- spam score e lartë.

Kjo e bën modelin më afër sistemeve reale.



---

# 5. Eksperimente numerike

Në këtë pjesë do të studiojmë si ndryshon renditja kur ndryshojmë:

1. faktorin e zbehjes \(d\),
2. rëndësinë e relevancës tekstuale,
3. ndëshkimin ndaj spam-it.

Kjo është analoge me analizën e ndjeshmërisë në modelim numerik.


In [ ]:

def final_ranking_with_params(query, pages, damping=0.85, alpha=0.15, beta=0.35, gamma=0.10, delta=0.15, epsilon=0.15, zeta=0.10):
    rel = relevance_scores(query, pages, doc_vectors, vocab_index, idf)
    rel_norm = normalize_dict(rel)

    ids_basic, pr_basic, _ = pagerank_basic(pages, damping=damping)
    pr_basic_dict = normalize_dict({pid: score for pid, score in zip(ids_basic, pr_basic)})

    ids_w, pr_weighted = pagerank_weighted(pages, personalization=rel_norm, damping=damping)
    pr_weighted_dict = normalize_dict({pid: score for pid, score in zip(ids_w, pr_weighted)})

    fresh, qual, beh, spam = page_feature_scores(pages)

    scores = {}
    for pid in pages:
        authority = 0.4 * pr_basic_dict[pid] + 0.6 * pr_weighted_dict[pid]
        scores[pid] = (
            alpha * authority
            + beta * rel_norm[pid]
            + gamma * fresh[pid]
            + delta * qual[pid]
            + epsilon * beh[pid]
            - zeta * spam[pid]
        )
    return scores

damping_values = [0.50, 0.70, 0.85, 0.95]
top_scores = {pid: [] for pid in pages}

for d in damping_values:
    s = final_ranking_with_params(query, pages, damping=d)
    s_norm = normalize_dict(s)
    for pid in pages:
        top_scores[pid].append(s_norm[pid])

plt.figure(figsize=(11, 6))
for pid, vals in top_scores.items():
    plt.plot(damping_values, vals, marker='o', label=pid)

plt.xlabel("Damping factor d")
plt.ylabel("Score finale e normalizuar")
plt.title("Ndjeshmëria e renditjes ndaj faktorit të zbehjes")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:

# Ndikimi i peshës së relevancës tekstuale
beta_values = np.linspace(0.10, 0.60, 6)
tracked = ["P1", "P3", "P6", "P7"]
beta_curves = {pid: [] for pid in tracked}

for beta in beta_values:
    # Rregullojmë alpha që shuma e disa peshave të mos rritet tepër
    alpha = 0.50 - beta if (0.50 - beta) > 0.05 else 0.05
    s = final_ranking_with_params(query, pages, alpha=alpha, beta=beta)
    s_norm = normalize_dict(s)
    for pid in tracked:
        beta_curves[pid].append(s_norm[pid])

plt.figure(figsize=(10, 6))
for pid, vals in beta_curves.items():
    plt.plot(beta_values, vals, marker='o', label=f"{pid} - {pages[pid]['title'][:24]}")

plt.xlabel("Pesha e relevancës tekstuale β")
plt.ylabel("Score finale e normalizuar")
plt.title("Ndikimi i β në renditjen e faqeve")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:

# Ndikimi i penalizimit të spam-it
zeta_values = np.linspace(0.0, 0.4, 9)
spam_impact = {pid: [] for pid in ["P6", "P7"]}

for zeta in zeta_values:
    s = final_ranking_with_params(query, pages, zeta=zeta)
    s_norm = normalize_dict(s)
    for pid in spam_impact:
        spam_impact[pid].append(s_norm[pid])

plt.figure(figsize=(9, 5))
for pid, vals in spam_impact.items():
    plt.plot(zeta_values, vals, marker='o', label=f"{pid} - {pages[pid]['title'][:28]}")

plt.xlabel("Pesha e penalizimit të spam-it ζ")
plt.ylabel("Score finale e normalizuar")
plt.title("Ndikimi i penalizimit të spam-it")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



## 5.1 Koment mbi realizmin e modelit

Modeli i përmirësuar është më realist sepse:

- dallon **autoritetin global** nga **relevanca lokale ndaj pyetjes**;
- lejon ndërthurjen e informacionit **strukturor**, **tekstual** dhe **sjellor**;
- penalizon faqe me sjellje tipike spam-i;
- mund të zgjerohet lehtësisht me veçori të tjera.

Në sisteme reale, përdoren edhe komponentë më të avancuar:
- modele gjuhësore,
- *learning to rank*,
- sinjale semantike dense,
- personalizim sipas përdoruesit,
- sinjale nga sesionet dhe konteksti.

Megjithatë, skeleti matematik dhe algoritmik i ndërtuar këtu është një bazë e shkëlqyer didaktike.



---

# 6. Ushtrime për studentët

## 6.1 Ushtrimi 1 — Implementoni HITS

Implementoni algoritmin **HITS** dhe llogarisni:
- score-n e **hub**,
- score-n e **authority**.

Pastaj krahasoni rezultatet me PageRank.

## 6.2 Ushtrimi 2 — Krahasoni PageRank klasik me Personalized PageRank

Detyra:
- përdorni të paktën tri pyetje të ndryshme;
- llogarisni renditjen me PageRank klasik;
- llogarisni renditjen me Personalized PageRank;
- interpretoni ndryshimet.

## 6.3 Ushtrimi 3 — Matni kompleksitetin empirik

Studentët mund të matin:
- kohën e ekzekutimit për madhësi të ndryshme të grafit;
- varësinë nga numri i nyjeve \(|V|\);
- varësinë nga numri i lidhjeve \(|E|\);
- numrin e iterimeve deri në konvergjencë.

## 6.4 Ushtrimi 4 — Përmirësoni modelin

Zgjeroni score-n finale duke shtuar të paktën një nga këto:
1. ngjashmëri semantike më të avancuar;
2. penalizim për fermat e lidhjeve;
3. prioritet tematik sipas fushës shkencore;
4. personalizim sipas përdoruesit.

## 6.5 Ushtrimi reflektues

Shpjegoni pse një faqe me shumë lidhje hyrëse mund të mos jetë domosdoshmërisht rezultati më i mirë për një pyetje të caktuar.



---

# 7. Përfundime

Në këtë notebook pamë se:

- renditja e faqeve mund të modelohet si problem në një graf;
- **PageRank** është një model elegant dhe probabilistik i autoritetit global;
- një algoritëm realist i renditjes duhet të kombinojë shumë sinjale;
- **relevanca**, **freskia**, **cilësia**, **sjellja e përdoruesit** dhe **anti-spam-i** janë komponentë thelbësorë;
- analiza e ndjeshmërisë ndaj parametrave është pjesë e domosdoshme e modelimit.

Mesazhi kryesor është ky:

> në sisteme reale të informacionit, renditja e faqeve nuk është thjesht problem i numërimit të lidhjeve, por problem i integrimit të shumë burimeve informacioni në një model të qëndrueshëm dhe të interpretuar mirë.

Ky këndvështrim e bën temën veçanërisht të rëndësishme për një kurs të **Modelimit në Fizikë**, sepse:
- grafet,
- proceset stokastike,
- konvergjenca iterative,
- dhe kombinimi i sinjaleve
janë motive të përbashkëta si në informatikë ashtu edhe në modelimin shkencor.
